In [ ]:
import os
import time
import numpy as np
import pandas as pd
from lightgbm import LGBMRegressor
from pathlib import Path


# -------------------------------------------------
# Paths
# -------------------------------------------------
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "code":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "data"
RESULTS_DIR = PROJECT_ROOT / "results" / "intermediate"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

TERRAIN_PATH = DATA_DIR / "weightedTerrainData.csv"
OUT_LONG = RESULTS_DIR / "xgboost_table23_long.csv"
OUT_SUMM = RESULTS_DIR / "xgboost_table23_summary.csv"

print("Working directory:", Path.cwd())
print("Project root:", PROJECT_ROOT)
print("Terrain path:", TERRAIN_PATH)
print("Terrain exists:", TERRAIN_PATH.exists())

if not TERRAIN_PATH.exists():
    raise FileNotFoundError(f"Terrain file not found: {TERRAIN_PATH}")
# -------------------------------------------------
# IDs
# -------------------------------------------------
TURBINE_IDS = list(range(1, 67))
TESTSET_2018 = list(range(1, 47)) + [48, 49, 50, 52] + list(range(54, 61)) + list(range(62, 67))

# -------------------------------------------------
# Terrain (cols 2:4, NO scaling to mimic your latest code)
# -------------------------------------------------
terrain_df = pd.read_csv(TERRAIN_PATH)
terrain_cols = terrain_df.columns[1:4]
terrain_mat = terrain_df.loc[:65, terrain_cols].values  # row i-1 = turbine i

# -------------------------------------------------
# Helpers
# -------------------------------------------------
def load_turbine_csv(tid, year):
    f = DATA_DIR / f"Turbine{tid}_{year}.csv"
    if not f.exists():
        return None
    df = pd.read_csv(f)
    for c in ["wind_speed", "temperature", "power"]:
        df[c] = pd.to_numeric(df[c], errors="coerce")
    return df

def rmse(yhat, y):
    yhat = np.asarray(yhat, dtype=float)
    y = np.asarray(y, dtype=float)
    m = np.isfinite(yhat) & np.isfinite(y)
    if not np.any(m):
        return np.nan
    return float(np.sqrt(np.mean((yhat[m] - y[m])**2)))

# -------------------------------------------------
# Cache data
# -------------------------------------------------
data_cache = {}
for tid in TURBINE_IDS:
    for yr in (2017, 2018):
        data_cache[(tid, yr)] = load_turbine_csv(tid, yr)

# -------------------------------------------------
# Build full 2017 pool
# -------------------------------------------------
X2017_list, y2017_list, tid2017_list = [], [], []

for tid in TURBINE_IDS:
    df = data_cache[(tid, 2017)]
    if df is None:
        continue

    X = df[["wind_speed", "temperature"]].values
    y = df["power"].values
    t = np.full(len(y), tid, dtype=int)

    X2017_list.append(X)
    y2017_list.append(y)
    tid2017_list.append(t)

X2017 = np.vstack(X2017_list)
y2017 = np.concatenate(y2017_list)
tid2017 = np.concatenate(tid2017_list)

S2017 = terrain_mat[tid2017 - 1]

# -------------------------------------------------
# Fixed LightGBM model
# -------------------------------------------------
def fit_lgbm(X_train, y_train, X_test, seed):
    model = LGBMRegressor(
        objective="regression",
        n_estimators=200,
        learning_rate=0.1,
        max_depth=8,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=seed,
        n_jobs=-1
    )
    model.fit(X_train, y_train)
    return model.predict(X_test)

# -------------------------------------------------
# Main loops: test on 2017 and 2018
# -------------------------------------------------
results = []

for year in (2017, 2018):
    test_ids = TURBINE_IDS if year == 2017 else TESTSET_2018

    for i in test_ids:
        print(f"Evaluating Turbine {i}, Year {year}")

        df_test = data_cache[(i, year)]
        if df_test is None:
            continue

        X_test = df_test[["wind_speed", "temperature"]].values
        y_test = df_test["power"].values

        # Terrain for test turbine
        S_test = np.tile(terrain_mat[i-1], (len(X_test), 1))

        # LOO training from 2017 pool
        mask = tid2017 != i
        X_train = X2017[mask]
        y_train = y2017[mask]
        S_train = S2017[mask]

        # -----------------
        # LGBM(x)
        # -----------------
        t0 = time.time()
        pred_x = fit_lgbm(X_train, y_train, X_test, seed=i)
        t1 = time.time()
        results.append({
            "Method": "LGBM(x)",
            "Turbine": i,
            "Year": year,
            "RMSE": rmse(pred_x, y_test),
            "Runtime": round(t1 - t0, 4)
        })

        # -----------------
        # LGBM(x+s)
        # -----------------
        Xs_train = np.hstack([X_train, S_train])
        Xs_test  = np.hstack([X_test, S_test])

        t0 = time.time()
        pred_xs = fit_lgbm(Xs_train, y_train, Xs_test, seed=i)
        t1 = time.time()
        results.append({
            "Method": "LGBM(x+s)",
            "Turbine": i,
            "Year": year,
            "RMSE": rmse(pred_xs, y_test),
            "Runtime": round(t1 - t0, 4)
        })

# -------------------------------------------------
# Save outputs
# -------------------------------------------------
results_long = pd.DataFrame(results)
results_long.to_csv(OUT_LONG, index=False)

summary = (
    results_long
    .groupby(["Method", "Year"], as_index=False)
    .agg(RMSE=("RMSE", "mean"),
         Runtime=("Runtime", "mean"))
)

summary_wide = summary.pivot(index="Method", columns="Year")
summary_wide.columns = [f"{a}{b}" for a, b in summary_wide.columns]
summary_wide.reset_index().to_csv(OUT_SUMM, index=False)

print("Saved results:")
print(" -", OUT_LONG)
print(" -", OUT_SUMM)

# -------------------------------------------------
# Update final results.csv : Table 2 and Table 3, XGBoost row
# -------------------------------------------------
import sys
sys.path.insert(0, str(PROJECT_ROOT / "code"))
from update_final_results import update_final_results

import numpy as np

summary_wide = summary_wide.reset_index()

def get_val(method_name, col_name):
    row = summary_wide.loc[summary_wide["Method"] == method_name, col_name]
    return row.iloc[0] if len(row) > 0 else np.nan

update_final_results(
    method="XGBoost",
    table_id="Table 2",
    rmse=get_val("LGBM(x)", "RMSE2017"),
    nlpd=np.nan,
    runtime=get_val("LGBM(x)", "Runtime2017")
)

update_final_results(
    method="XGBoost",
    table_id="Table 3",
    rmse=get_val("LGBM(x)", "RMSE2018"),
    nlpd=np.nan,
    runtime=get_val("LGBM(x)", "Runtime2018")
)